# 🎨 VQ-VAE Part B: Autoregressive Image Generation

## From Tokens to Generation

After VQ-VAE training, every image is a **discrete token sequence** — just like text.

The key insight: if we can model `P(token_i | token_1, ..., token_{i-1})`, we can **sample** token sequences and decode them back to images.

**The pipeline:**
```
Image  →  VQ-VAE Encoder  →  Token Grid  →  Flatten  →  [42, 17, 305, 8, ...]  →  Transformer learns P(next token)
                                                                                              ↓
New Image  ←  VQ-VAE Decoder  ←  Token Grid  ←  Reshape  ←  Sample new token sequence
```

This is exactly GPT-style generation — but for images. The Transformer (e.g. PixelSNAIL, DALL-E, ImageGPT) autoregressively predicts each spatial token given all previous ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

compression = 8  # VQ-VAE spatial compression factor
image_sizes = [64, 256, 1024]
token_counts = [(s // compression) ** 2 for s in image_sizes]

print("Image Size  | Token Grid       | Token Count")
print("-" * 46)
for s, t in zip(image_sizes, token_counts):
    grid = s // compression
    print(f"{s:>6}x{s:<6} | {grid:>4}x{grid:<4} tokens    | {t:>7,}")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f"{s}x{s}" for s in image_sizes], token_counts, color=["#4CAF50", "#FF9800", "#F44336"])
for bar, t in zip(bars, token_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f"{t:,}", ha="center", fontsize=10)
ax.set_title("Token Count per Image Size (8x compression)", fontsize=13)
ax.set_xlabel("Image Resolution")
ax.set_ylabel("Number of Tokens")
plt.tight_layout()
plt.show()

In [ ]:
import torch

# Simulate a 8x8 token grid (for a 64x64 image with 8x compression)
H, W = 8, 8
token_grid = torch.randint(0, 512, (H, W))  # 512-entry codebook

# Flatten 2D grid -> 1D sequence (raster order: left-to-right, top-to-bottom)
flat_tokens = token_grid.flatten()  # shape: (64,)

# Autoregressive training format: input is [t0..t_{N-2}], target is [t1..t_{N-1}]
input_seq  = flat_tokens[:-1]  # all tokens except last
target_seq = flat_tokens[1:]   # all tokens except first (shifted by 1)

print(f"Token grid shape : {tuple(token_grid.shape)}")
print(f"Flat sequence    : {flat_tokens.shape}  (length {len(flat_tokens)})")
print(f"Input  sequence  : {input_seq.shape}")
print(f"Target sequence  : {target_seq.shape}")
print()
print("First 8 input tokens :", input_seq[:8].tolist())
print("First 8 target tokens:", target_seq[:8].tolist())
print("\nAt each step, model predicts target[i] given input[0..i] -> cross-entropy loss.")

## Complexity Comparison: Autoregressive vs Diffusion

Both approaches work well for image generation, but they have very different computational profiles.

| Dimension            | Autoregressive (VQ-VAE + Transformer)          | Diffusion (DDPM / Stable Diffusion)              |
|----------------------|------------------------------------------------|--------------------------------------------------|
| **Training cost**    | O(N²) per sample — Transformer self-attention  | O(T · N²) — T denoising steps × attention       |
| **Inference cost**   | O(N²) — sequential, N token steps             | O(T · N²) — T denoising passes (T ≈ 20–1000)    |
| **Parallelism**      | Sequential at inference (tokens one by one)    | Parallel within each denoising step              |
| **Resolution scaling** | Tokens grow as (H/f)² — expensive at 1024px | Latent diffusion keeps cost manageable           |
| **Conditioning**     | Natural: prepend class/text tokens             | Cross-attention at each denoising step           |
| **Sample quality**   | Sharp, globally consistent                     | Very high quality, continuous                    |
| **Best for**         | Discrete outputs, small-medium resolution      | High-res photorealistic images, large-scale gen  |

**Rule of thumb:** Autoregressive wins when you need discrete, structured outputs (e.g. tokens, code, symbolic images). Diffusion wins at high-resolution photorealism. Hybrid systems (VQ-VAE tokenizer + diffusion in token space) combine both.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pixels = np.array([64, 128, 256, 384, 512])
N = (pixels // 8) ** 2          # token count (8x spatial compression)
T = 50                           # diffusion steps (DDIM-style)

ar_cost   = N ** 2               # O(N^2) — Transformer self-attention
diff_cost = T * N ** 2           # O(T * N^2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pixels, ar_cost / ar_cost[0],   "b-o", label="Autoregressive O(N²)")
ax.plot(pixels, diff_cost / ar_cost[0], "r-s", label=f"Diffusion O(T·N²), T={T}")
ax.set_xlabel("Image size (pixels)", fontsize=11)
ax.set_ylabel("Relative compute (normalized)", fontsize=11)
ax.set_title("Inference Complexity: Autoregressive vs Diffusion", fontsize=13)
ax.legend(fontsize=10)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## System Design: High-Resolution Image Synthesis

### Full Pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│                    TRAINING TIME                                    │
│                                                                     │
│  Images (256x256) ──► VQ-VAE Encoder ──► 32x32 Token Grid          │
│                            │                     │                  │
│                     (learns codebook)    Flatten to 1024 tokens     │
│                                                  │                  │
│                                        Transformer (GPT-style)      │
│                                        learns P(next token)         │
└─────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────┐
│                    INFERENCE TIME                                   │
│                                                                     │
│  Text prompt / class label                                          │
│         │                                                           │
│         ▼                                                           │
│  [BOS] + condition tokens                                           │
│         │                                                           │
│         ▼                                                           │
│  Transformer autoregressively samples 1024 image tokens             │
│         │                                                           │
│         ▼                                                           │
│  Reshape 1024 tokens ──► 32x32 grid                                 │
│         │                                                           │
│         ▼                                                           │
│  VQ-VAE Decoder ──► 256x256 image                                   │
│         │                                                           │
│         ▼                                                           │
│  Super-Resolution Service (optional) ──► 1024x1024 final output     │
└─────────────────────────────────────────────────────────────────────┘
```

### Interview Cheat Sheet

| Question | Answer |
|---|---|
| Why use VQ-VAE before Transformer? | Compresses image ~64x, making sequence length tractable for attention |
| Why discrete tokens (not continuous)? | Discrete = well-defined next-token prediction with cross-entropy |
| How do you condition on text? | Prepend text tokens to image token sequence (same Transformer) |
| How does super-resolution fit in? | Separate model upscales the 256px output to 1024px+ |
| Bottleneck at scale? | Transformer attention O(N²) — use sliding window or sparse attention for very long sequences |
| How to speed up inference? | Top-k / nucleus sampling, smaller codebook, parallel decoding tricks (MaskGIT) |

## Quiz

**Q1: A 256x256 image is tokenized with an 8x compression VQ-VAE. How many tokens does the Transformer need to generate, and what is the self-attention cost relative to a 128x128 image?**

<details>
<summary>Show answer</summary>

A 256x256 image with 8x compression → 32x32 token grid = **1024 tokens**.  
A 128x128 image → 16x16 = 256 tokens.  
Self-attention is O(N²), so cost ratio = (1024/256)² = **16x more expensive** for the 256px image.

</details>

---

**Q2: Why is the VQ-VAE codebook size an important design choice? What happens if it is too small or too large?**

<details>
<summary>Show answer</summary>

The codebook size is the vocabulary of the image "language".  
- **Too small** (e.g. 64): High quantization error — reconstructed images are blurry. The Transformer also has an easy job (small vocab), but the decoder cannot reproduce fine detail.  
- **Too large** (e.g. 16384+): Low quantization error, but codebook collapse is a risk (many entries never used). The Transformer has a harder classification problem per step.  
- **Typical sweet spot**: 512–8192. DALL-E used 8192; VQ-VAE-2 used 512 per level.

</details>

---

**Q3: MaskGIT is an alternative to left-to-right autoregressive generation. What is its key idea and why is it faster at inference?**

<details>
<summary>Show answer</summary>

MaskGIT uses a **masked token prediction** approach (like BERT, not GPT).  
- At inference, start with all tokens masked.  
- Each iteration, predict all masked tokens **in parallel**, then keep only the most confident ones and re-mask the rest.  
- Repeat for ~8–16 iterations.  
This is **much faster** than generating N tokens one-by-one (N sequential steps → ~log(N) parallel steps), while maintaining comparable quality.

</details>